# Deploy Tool Backend Technical Requirements

This notebook defines the backend technical requirements and implementation blueprint for the FastAPI deployment server.


## 1. Project Structure

```
backend/
├── pyproject.toml
├── .env.example
├── app/
│   ├── __init__.py
│   ├── main.py
│   ├── config.py
│   ├── models.py
│   ├── routers/
│   │   ├── __init__.py
│   │   ├── deploy.py
│   │   ├── branches.py
│   │   ├── status.py
│   │   └── websocket.py
│   ├── services/
│   │   ├── __init__.py
│   │   ├── repo_manager.py
│   │   ├── deployer.py
│   │   └── log_streamer.py
│   └── utils/
│       ├── __init__.py
│       └── validation.py
└── backend_requirements.ipynb
```


## 2. API Endpoints

### `GET /api/status`
- Response schema:
```json
{
  "status": "ok",
  "repos": {
    "nbn-daemon": true,
    "unity": true
  },
  "docker": true
}
```

### `GET /api/branches/{repo}`
- `repo` allowed: `nbn-daemon`, `unity`
- Fetches from origin before listing branches
- Response schema:
```json
{
  "branches": ["main", "feature/x"]
}
```

### `POST /api/deploy`
- Request schema:
```json
{
  "repo": "nbn-daemon",
  "branch": "feature/x",
  "filerIP": "10.0.0.10"
}
```
- Immediate response:
```json
{
  "deploymentId": "uuid",
  "status": "started"
}
```

### `GET /api/deployments/{id}`
- Response schema:
```json
{
  "id": "uuid",
  "repo": "unity",
  "branch": "feature/x",
  "filerIP": "10.0.0.10",
  "status": "running",
  "exitCode": null,
  "startedAt": "ISO8601",
  "completedAt": null
}
```

### `WS /api/ws/logs/{deploymentId}`
- Stream message schema:
```json
{
  "type": "stdout",
  "line": "...",
  "timestamp": "ISO8601"
}
```
- Completion message:
```json
{
  "type": "system",
  "line": "Deployment finished with exit code X",
  "timestamp": "ISO8601",
  "done": true
}
```


## 3. WebSocket Protocol

- Channel key: `deploymentId`
- Multiple clients may subscribe to one deployment
- Server broadcasts JSON messages to all connected clients
- Message `type` values: `stdout`, `stderr`, `system`
- `done=true` only on final completion event


## 4. Repo Management Module

`RepoManager` responsibilities:
1. `ensure_cloned(repo)`
   - Clone if missing to `~/.deploy_tool/repos/<repo>`
   - Configure PAT-authenticated origin URL
2. `prepare_branch(repo, branch)`
   - `git fetch origin`
   - Validate `origin/<branch>` exists
   - `git reset --hard HEAD` and `git clean -fd`
   - Checkout branch (create tracking if needed)
   - `git rebase origin/<branch>`
3. `list_branches(repo)`
   - `git fetch origin`
   - `git branch -r` parse and strip `origin/` + `HEAD`

All git calls use `asyncio.create_subprocess_exec`.


## 5. Deployment Engine

`Deployer` responsibilities:
- Run subprocesses with timeout (default 1800 seconds)
- Capture stdout/stderr concurrently line-by-line
- Broadcast each line through log streamer
- Kill process on timeout and mark deployment failed

### nbn-daemon flow
- Working directory: `~/.deploy_tool/repos/nbn-daemon`
- Command: `make deploy-rpm FILER=<filerIP>`

### Unity flow
1. `python python/tools/sync-dev.py --restart <filerIP>`
2. SSH post-sync on port 222:
   - Delete `.pyc`/`.pyo`
   - Restart `filer-route-http`, `qman`, `httpd`


## 6. nbn-daemon Deploy Flow Details

1. Acquire `nbn-daemon` lock
2. Prepare branch (fetch/reset/checkout/rebase)
3. Run `make deploy-rpm FILER=<ip>`
4. Stream logs to websocket
5. Set final status + exit code
6. Release lock


## 7. Unity Deploy Flow Details

1. Acquire `unity` lock
2. Prepare branch (fetch/reset/checkout/rebase)
3. Run `python python/tools/sync-dev.py --restart <ip>`
4. Run SSH post-sync command on port `222`:
   ```bash
   ssh -p222 root@<ip> "find /opt/nasuni/lib/nasuni/ -name '*.pyc' -delete && find /opt/nasuni/lib/nasuni/ -name '*.pyo' -delete && systemctl restart filer-route-http qman && systemctl restart httpd"
   ```
5. Stream logs and final result
6. Release lock


## 8. Concurrency Model

- Keep two `asyncio.Lock` instances keyed by repo
- One deployment per repo at a time
- Different repos can deploy concurrently
- `deployments: dict[str, DeploymentState]` stores active + completed state in memory


## 9. Configuration Schema

```python
class Settings(BaseSettings):
    github_pat: str
    repos_base_path: str = "~/.deploy_tool/repos"
    nbn_daemon_repo_url: str = "https://github.com/nasuni/nbn-daemon.git"
    unity_repo_url: str = "https://github.com/nasuni/unity.git"
    deploy_timeout_seconds: int = 1800
    server_port: int = 8000
```

Environment source: `.env` and process environment variables.


## 10. Input Validation Rules

- Repo: must be exactly `nbn-daemon` or `unity`
- Branch: validated with `git check-ref-format --branch <name>`
- Filer IP: validated with `ipaddress.IPv4Address(...)`
- Invalid inputs return HTTP 400


## 11. Error Handling

- Git failures -> HTTP 4xx/5xx with detailed message
- Missing deployment id -> 404
- Deployment subprocess non-zero exit -> status `failed` + `exitCode`
- Timeout -> process killed, status `failed`, system log emitted
- WebSocket disconnects are handled without crashing deployment


## 12. Python Dependencies

`pyproject.toml` dependencies:
- `fastapi>=0.115`
- `uvicorn[standard]>=0.30`
- `pydantic-settings>=2.0`
- `websockets>=12.0`

Dev dependency:
- `pytest>=8.0` (for backend tests)
